In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# Agent Laboratory

The book's example (pp. 317-324) is not a program but a tour of excerpts from [Agent Laboratory](https://github.com/SamuelSchmidgall/AgentLaboratory), Samuel Schmidgall's MIT-licensed framework. It runs research as a team of LLM agents: a literature review, then experiments, then a written report, with a professor, a postdoc, a PhD student, ML and software engineers, and three reviewers. The real system needs arXiv, Hugging Face and LaTeX, and is too large to run here. So this notebook runs **the book's excerpts**, completed from upstream's `agents.py`, on one toy topic.

| The book | Here |
|---|---|
| `get_score` cut off after its output format | completed from upstream: parse the JSON, weigh the ratings into one score. Upstream's long NeurIPS review form is left out, as in the book |
| `query_model`, `BaseAgent`, `extract_json_between_markers` not shown | a short `query_model` on the OpenAI client, a shortened `BaseAgent`, and upstream's parser |
| `model="gpt4omini"` | upstream's internal alias; `MODEL`, set by each model cell |
| ML and software engineer prompts as bare strings | system prompts of a short exchange: the engineer directs, the ML engineer writes code |
| rating scales that name words as well as numbers | one sentence added: every rating must be a number (see below) |
| no call to any of them | `run_agent_lab`: plan, then data code, then reviews, then a README |

**What is not real.** The topic and both reports are written for this notebook. No experiment runs and no literature is searched, so the report's accuracy figures are **invented**, and the ML engineer's code is only checked for valid syntax: it is never executed, and scikit-learn is not a dependency here.

**Three things in the excerpts to look at**, all kept as written:

- `PostdocAgent.context` has a stray comma, so it returns a **tuple**, not text. The first line of output shows it, and the model reads that tuple's `repr`.
- `get_score` has `attempts=3`, but its `except` returns on the first failure, so it **never retries**. On the local model, one answer of `"Confidence": "high"` made `int()` raise and discarded the whole review, until the sentence above was added. The scales invite that answer: "A rating from 1 to 5 (low, medium, high, very high, absolute)".
- The README the professor writes can claim things nobody did: an MIT license nobody chose, or, in one run, code that "was used" for the analysis, when nothing ran. It also repeats the report's invented figures as results. Compare it with what actually happened.

The next cell holds the excerpts and the run helper, with no model. The OpenAI cell and the Ollama cell each bind `client` and `MODEL`, so either can run on its own.

In [3]:
# Book listing: Chapter 21, Hands-On Code Example, pp. 319-323: excerpts of Agent Laboratory's
# agents.py (https://github.com/SamuelSchmidgall/AgentLaboratory, MIT License, Copyright (c) 2025
# Samuel Schmidgall). What the book cuts is completed from that file: the end of get_score,
# extract_json_between_markers, and a shortened BaseAgent.
# This cell holds no model. query_model reads the globals `client` and `MODEL` when it runs; the
# OpenAI cell and the Ollama cell below each bind those two and call run_agent_lab.
import ast
import json
import re


def query_model(model_str, system_prompt, prompt, temp=0.0, openai_api_key=None):
    """Stands in for upstream's query_model. model_str and the key are ignored: see MODEL, client."""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=temp,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content


def extract_json_between_markers(llm_output):
    # Regular expression pattern to find JSON content between ```json and ```
    json_pattern = r"```json(.*?)```"
    matches = re.findall(json_pattern, llm_output, re.DOTALL)

    if not matches:
        # Fallback: Try to find any JSON-like content in the output
        json_pattern = r"\{.*?\}"
        matches = re.findall(json_pattern, llm_output, re.DOTALL)

    for json_string in matches:
        json_string = json_string.strip()
        try:
            parsed_json = json.loads(json_string)
            return parsed_json
        except json.JSONDecodeError:
            # Attempt to fix common JSON issues
            try:
                # Remove invalid control characters
                json_string_clean = re.sub(r"[\x00-\x1F\x7F]", "", json_string)
                parsed_json = json.loads(json_string_clean)
                return parsed_json
            except json.JSONDecodeError:
                continue  # Try next match

    return None  # No valid JSON found


class ReviewersAgent:
    def __init__(self, model="gpt-4o-mini", notes=None, openai_api_key=None):
        if notes is None: self.notes = []
        else: self.notes = notes
        self.model = model
        self.openai_api_key = openai_api_key

    def inference(self, plan, report):
        reviewer_1 = "You are a harsh but fair reviewer and expect good experiments that lead to insights for the research topic."
        review_1 = get_score(outlined_plan=plan, latex=report, reward_model_llm=self.model, reviewer_type=reviewer_1, openai_api_key=self.openai_api_key)

        reviewer_2 = "You are a harsh and critical but fair reviewer who is looking for an idea that would be impactful in the field."
        review_2 = get_score(outlined_plan=plan, latex=report, reward_model_llm=self.model, reviewer_type=reviewer_2, openai_api_key=self.openai_api_key)

        reviewer_3 = "You are a harsh but fair open-minded reviewer that is looking for novel ideas that have not been proposed before."
        review_3 = get_score(outlined_plan=plan, latex=report, reward_model_llm=self.model, reviewer_type=reviewer_3, openai_api_key=self.openai_api_key)

        return f"Reviewer #1:\n{review_1}, \nReviewer #2:\n{review_2}, \nReviewer #3:\n{review_3}"


def get_score(outlined_plan, latex, reward_model_llm, reviewer_type=None, attempts=3, openai_api_key=None):
    e = str()
    for _attempt in range(attempts):
        try:
            # todo: have a reward function here
            # template inherited from the AI Scientist (good work on this prompt Sakana AI team :D)
            template_instructions = """
            Respond in the following format:

            THOUGHT:
            <THOUGHT>

            REVIEW JSON:
            ```json
            <JSON>
            ```

            In <THOUGHT>, first briefly discuss your intuitions and reasoning for the evaluation.
            Detail your high-level arguments, necessary choices and desired outcomes of the review.
            Do not make generic comments here, but be specific to your current paper.
            Treat this as the note-taking phase of your review.

            In <JSON>, provide the review in JSON format with the following fields in the order:
            - "Summary": A summary of the paper content and its contributions.
            - "Strengths": A list of strengths of the paper.
            - "Weaknesses": A list of weaknesses of the paper.
            - "Originality": A rating from 1 to 4 (low, medium, high, very high).
            - "Quality": A rating from 1 to 4 (low, medium, high, very high).
            - "Clarity": A rating from 1 to 4 (low, medium, high, very high).
            - "Significance": A rating from 1 to 4 (low, medium, high, very high).
            - "Questions": A set of clarifying questions to be answered by the paper authors.
            - "Limitations": A set of limitations and potential negative societal impacts of the work.
            - "Ethical Concerns": A boolean value indicating whether there are ethical concerns.
            - "Soundness": A rating from 1 to 4 (poor, fair, good, excellent).
            - "Presentation": A rating from 1 to 4 (poor, fair, good, excellent).
            - "Contribution": A rating from 1 to 4 (poor, fair, good, excellent).
            - "Overall": A rating from 1 to 10 (very strong reject to award quality).
            - "Confidence": A rating from 1 to 5 (low, medium, high, very high, absolute).
            - "Decision": A decision that has to be one of the following: Accept, Reject.

            For the "Decision" field, don't use Weak Accept, Borderline Accept, Borderline Reject, or Strong Reject. Instead, only use Accept or Reject.
            This JSON will be automatically parsed, so ensure the format is precise.
            Every rating must be a number, not a word.
            """
            if reviewer_type is None: reviewer_type = ""
            # The last line of the template is added: the rating scales name words as well as numbers,
            # and int("high") would discard the review.
            # Upstream also appends a long NeurIPS review form here; the book omits it, and so does this cell.
            sys_prompt = (
                "You are an AI researcher who is reviewing a paper that was submitted to a prestigious ML venue. "
                f"Be critical and cautious in your decision. {reviewer_type}\n"
            ) + template_instructions
            scoring = query_model(
                model_str=f"{reward_model_llm}",
                system_prompt=sys_prompt,
                openai_api_key=openai_api_key,
                prompt=(
                    f"Outlined in the following text is the research plan that the machine learning engineer was tasked with building: {outlined_plan}\n\n"
                    f"The following text is the research latex that the model produced: \n{latex}\n\n"), temp=0.0)
            review_json = extract_json_between_markers(scoring)

            overall = int(review_json["Overall"]) / 10
            soundness = int(review_json["Soundness"]) / 4
            confidence = int(review_json["Confidence"]) / 5
            contribution = int(review_json["Contribution"]) / 4
            presentation = int(review_json["Presentation"]) / 4
            clarity = int(review_json["Clarity"]) / 4
            originality = int(review_json["Originality"]) / 4
            quality = int(review_json["Quality"]) / 4
            significance = int(review_json["Significance"]) / 4

            clarity_weight = 0.1
            quality_weight = 0.1
            overall_weight = 1.0
            soundness_weight = 0.1
            confidence_weight = 0.1
            originality_weight = 0.1
            significance_weight = 0.1
            contribution_weight = 0.4
            presentation_weight = 0.2

            # max possible
            max_score = (
                clarity_weight + quality_weight + overall_weight + soundness_weight + confidence_weight + originality_weight + significance_weight + contribution_weight + presentation_weight)

            performance = ((
               soundness_weight * soundness + presentation_weight * presentation + confidence_weight * confidence + contribution_weight * contribution + overall_weight * overall + originality_weight * originality + significance * significance_weight + clarity_weight * clarity + quality_weight * quality) / max_score) * 10
            return performance, f"The performance of your submission is: {performance}" + scoring, True
        except Exception as e:
            print(e)
            return None, str(e), False
    return 0, e


class BaseAgent:
    """The part of upstream's BaseAgent these two excerpts use."""

    def __init__(self, model="gpt-4o-mini", notes=None, max_steps=100, openai_api_key=None):
        if notes is None: self.notes = []
        else: self.notes = notes
        self.max_steps = max_steps
        self.model = model
        self.phases = []
        self.plan = str()
        self.report = str()
        self.history = list()
        self.exp_results = str()
        self.dataset_code = str()
        self.results_code = str()
        self.lit_review_sum = str()
        self.openai_api_key = openai_api_key
        self.second_round = False


class ProfessorAgent(BaseAgent):
    def __init__(self, model="gpt4omini", notes=None, max_steps=100, openai_api_key=None):
        super().__init__(model, notes, max_steps, openai_api_key)
        self.phases = ["report writing"]

    def generate_readme(self):
        sys_prompt = f"""You are {self.role_description()} \n Here is the written paper \n{self.report}. Task instructions: Your goal is to integrate all of the knowledge, code, reports, and notes provided to you and generate a readme.md for a github repository."""
        history_str = "\n".join([_[1] for _ in self.history])
        prompt = (
            f"""History: {history_str}\n{'~' * 10}\n"""
            f"Please produce the readme below in markdown:\n")
        model_resp = query_model(model_str=self.model, system_prompt=sys_prompt, prompt=prompt, openai_api_key=self.openai_api_key)
        return model_resp.replace("```markdown", "")

    def role_description(self):
        return "a computer science professor at a top university."


class PostdocAgent(BaseAgent):
    def __init__(self, model="gpt4omini", notes=None, max_steps=100, openai_api_key=None):
        super().__init__(model, notes, max_steps, openai_api_key)
        self.phases = ["plan formulation", "results interpretation"]

    def context(self, phase):
        sr_str = str()
        if self.second_round:
            sr_str = (
                f"The following are results from the previous experiments\n",
                f"Previous Experiment code: {self.prev_results_code}\n"
                f"Previous Results: {self.prev_exp_results}\n"
                f"Previous Interpretation of results: {self.prev_interpretation}\n"
                f"Previous Report: {self.prev_report}\n"
                f"{self.reviewer_response}\n\n\n"
            )
        if phase == "plan formulation":
            return (
                sr_str,
                f"Current Literature Review: {self.lit_review_sum}",
            )
        elif phase == "results interpretation":
            return (
                sr_str,
                f"Current Literature Review: {self.lit_review_sum}\n"
                f"Current Plan: {self.plan}\n"
                f"Current Dataset code: {self.dataset_code}\n"
                f"Current Experiment code: {self.results_code}\n"
                f"Current Results: {self.exp_results}"
            )
        return ""

    def role_description(self):
        return "a computer science postdoctoral student at a top university."


# The book prints these two as bare strings, parts of the ML and software engineers' phase prompts.
ML_ENGINEER_PROMPT = (
    "You are a machine learning engineer being directed by a PhD student who will help you write the code, and you can interact with them through dialogue.\n"
    "Your goal is to produce code that prepares the data for the provided experiment. You should aim for simple code to prepare the data, not complex code. You should integrate the provided literature review and the plan and come up with code to prepare data for this experiment.\n"
)

SW_ENGINEER_PROMPT = (
    "You are a software engineer directing a machine learning engineer, where the machine learning engineer will be writing the code, and you can interact with them through dialogue.\n"
    "Your goal is to help the ML engineer produce code that prepares the data for the provided experiment. You should aim for very simple code to prepare the data, not complex code. You should integrate the provided literature review and the plan and come up with code to prepare data for this experiment.\n"
)


# --- A toy run. Everything below is written for this notebook; it is not from Agent Laboratory. ---
TOPIC = "Does standardizing the features improve logistic regression on the Iris dataset?"
LITERATURE = "Gradient-based and regularized linear models are sensitive to feature scale, so standardization is common practice."
PLAN_TO_REVIEW = "Compare logistic regression with and without standardization on Iris, using 5-fold cross-validation."

# Two invented reports for the reviewers. No experiment was run; only their quality differs.
SOUND_REPORT = r"""
\section{Abstract} We compare logistic regression with and without standardization on Iris.
\section{Introduction} Linear models can be sensitive to feature scale.
\section{Methods} 5-fold stratified cross-validation over 150 samples, regularization C=1, the same folds for both models.
\section{Results} Without standardization, accuracy 0.973 (std 0.025). With standardization, 0.960 (std 0.033). The majority-class baseline is 0.333.
\section{Discussion} The difference is within one standard deviation, so we find no evidence of an effect. Iris features share a unit and a similar range, which limits generality.
"""

UNSOUND_REPORT = r"""
\section{Abstract} Our standardization breakthrough solves classification.
\section{Results} We trained on 10 samples and tested on the same 10 samples. Accuracy was 99.9%, which proves that standardization always works on all datasets.
"""


def review_summary(label: str, plan: str, report: str) -> None:
    """Prints each of the three reviewers' numbers; the book's inference returns them in one long string."""
    print(f"\n{label}:")
    for line in ReviewersAgent(model=MODEL).inference(plan, report).split("Reviewer #")[1:]:
        match = re.search(r"\((None|[\d.]+), ", line)
        decision = re.search(r'"Decision":\s*"(\w+)"', line)
        overall = re.search(r'"Overall":\s*(\d+)', line)
        print(f"  Reviewer {line[0]}: performance {match.group(1)[:5] if match else '?'}, "
              f"overall {overall.group(1) if overall else '?'}/10, decision {decision.group(1) if decision else '?'}")


def run_agent_lab() -> None:
    """One pass through the stages the book's excerpts come from."""
    postdoc = PostdocAgent(model=MODEL)
    postdoc.lit_review_sum = LITERATURE
    context = postdoc.context("plan formulation")
    print(f"--- PostdocAgent.context returns a {type(context).__name__}: {context!r}\n")
    plan = query_model(MODEL, f"You are {postdoc.role_description()} Produce a very simple experiment plan in three sentences.",
                       f"Topic: {TOPIC}\n{context}")
    print(f"--- Postdoc's plan:\n{plan}\n")

    direction = query_model(MODEL, SW_ENGINEER_PROMPT, f"Plan: {plan}\nLiterature: {LITERATURE}\nTell the ML engineer in two sentences what data preparation code to write.")
    print(f"--- Software engineer's direction:\n{direction}\n")
    answer = query_model(MODEL, ML_ENGINEER_PROMPT, f"Plan: {plan}\nThe software engineer says: {direction}\nReply with only the Python code, in a ```python block.")
    blocks = re.findall(r"```python(.*?)```", answer, re.DOTALL)
    code = blocks[0] if blocks else answer
    try:
        ast.parse(code)
        print(f"--- ML engineer's code ({len(code.splitlines())} lines) is valid Python. It is not executed here:\n{code.strip()}\n")
    except SyntaxError as error:
        print(f"--- ML engineer's code is not valid Python: {error}\n{code.strip()}\n")

    print("--- Three reviewers on two invented reports")
    review_summary("Sound report", PLAN_TO_REVIEW, SOUND_REPORT)
    review_summary("Unsound report", PLAN_TO_REVIEW, UNSOUND_REPORT)

    professor = ProfessorAgent(model=MODEL)
    professor.report = SOUND_REPORT
    professor.history = [(0, f"Plan: {PLAN_TO_REVIEW}"), (1, f"Data code:\n{code}")]
    print(f"\n--- Professor's README:\n{professor.generate_readme().strip()}")

In [4]:
# The excerpts on OpenAI, with the listing's default model.
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set; run the Ollama cell below instead.")

client = OpenAI()
MODEL = "gpt-4o-mini"

run_agent_lab()

--- PostdocAgent.context returns a tuple: ('', 'Current Literature Review: Gradient-based and regularized linear models are sensitive to feature scale, so standardization is common practice.')



--- Postdoc's plan:
To investigate whether standardizing the features improves logistic regression performance on the Iris dataset, I will first implement logistic regression using the raw features and evaluate its accuracy using cross-validation. Next, I will standardize the features using z-score normalization and re-evaluate the model's accuracy with the same cross-validation approach. Finally, I will compare the accuracy results from both models to determine the impact of feature standardization on logistic regression performance.



--- Software engineer's direction:
First, load the Iris dataset and split it into features and labels. Then, implement z-score normalization on the features to prepare them for the logistic regression model.



--- ML engineer's code (18 lines) is valid Python. It is not executed here:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Iris dataset
iris = load_iris()
X = iris.data  # Features
y = iris.target  # Labels

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Implement z-score normalization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

--- Three reviewers on two invented reports

Sound report:


  Reviewer 1: performance 4.568, overall 3/10, decision Reject
  Reviewer 2: performance 4.568, overall 3/10, decision Reject
  Reviewer 3: performance 3.999, overall 3/10, decision Reject

Unsound report:


  Reviewer 1: performance 2.409, overall 1/10, decision Reject
  Reviewer 2: performance 2.409, overall 1/10, decision Reject
  Reviewer 3: performance 2.409, overall 1/10, decision Reject



--- Professor's README:
# Logistic Regression on Iris Dataset

This repository contains a comparison of logistic regression performance with and without feature standardization on the Iris dataset. The analysis is conducted using 5-fold stratified cross-validation.

## Abstract

We compare logistic regression with and without standardization on the Iris dataset to evaluate the impact of feature scaling on model accuracy.

## Introduction

Linear models, such as logistic regression, can be sensitive to the scale of the features. This study aims to investigate whether standardizing the features affects the performance of logistic regression on the Iris dataset.

## Methods

- **Dataset**: Iris dataset containing 150 samples.
- **Cross-Validation**: 5-fold stratified cross-validation.
- **Regularization**: Regularization parameter C set to 1.
- **Model Comparison**: Both models (with and without standardization) are evaluated using the same folds.

## Results

- **Without Standardization

## Agent Laboratory + Ollama

The same code with the `OpenAI` client pointed at Ollama, as in chapter 16.

Compare the reviewers with the hosted cell. Both models rank the sound report above the unsound one, and both **reject both**, so the Accept/Reject decision separates nothing here; only the score does. And the scores do not share a scale: the same sound report gets about 3/10 from `gpt-4o-mini` and 5 to 6/10 from `qwen2.5:7b`. A score means something only next to other scores from the same reviewer.

In [5]:
# The same excerpts on a local model. Ollama serves an OpenAI-compatible API at /v1, so only
# base_url and the model name differ.
# Prerequisites:
#   ollama pull qwen2.5:7b
# Run cell 4 first (the shared cell under "# Agent Laboratory"); this cell needs nothing else: not the OpenAI cell, and no OpenAI key.
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # the key is ignored
MODEL = "qwen2.5:7b"

run_agent_lab()

--- PostdocAgent.context returns a tuple: ('', 'Current Literature Review: Gradient-based and regularized linear models are sensitive to feature scale, so standardization is common practice.')



--- Postdoc's plan:
Objective: To determine if standardizing features improves the performance of logistic regression on the Iris dataset.

Method: Apply logistic regression to the Iris dataset both with and without feature standardization, and compare the model accuracies using cross-validation.

Conclusion: Analyze the results to conclude whether standardization leads to better model performance in terms of accuracy and stability.



--- Software engineer's direction:
The ML engineer should load the Iris dataset, then standardize the feature columns using `StandardScaler` from `sklearn.preprocessing`. After that, split the data into training and testing sets, and prepare the datasets for model training and evaluation.



--- ML engineer's code (35 lines) is valid Python. It is not executed here:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the feature columns
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression model without scaling
logreg = LogisticRegression(solver='liblinear', random_state=42)
logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)
accuracy_no_scale = accuracy_score(y_test, y_pred)

# Train logistic regression model with scaling
logreg_scaled = LogisticRegres

  Reviewer 1: performance 6.272, overall 6/10, decision Reject
  Reviewer 2: performance 6.159, overall 6/10, decision Reject
  Reviewer 3: performance 4.909, overall 5/10, decision Reject

Unsound report:


  Reviewer 1: performance 2.340, overall 2/10, decision Reject
  Reviewer 2: performance 2.340, overall 2/10, decision Reject
  Reviewer 3: performance 3.227, overall 3/10, decision Reject



--- Professor's README:
# Logistic Regression Comparison with and without Standardization on Iris Dataset

## Overview
This repository contains the code and results for comparing the performance of logistic regression models with and without standardization on the Iris dataset. The comparison is conducted using 5-fold stratified cross-validation.

## Motivation
Linear models can be sensitive to feature scale. This experiment aims to determine if standardizing the features before training a logistic regression model improves its performance on the Iris dataset.

## Data and Methods
- **Dataset**: Iris dataset from scikit-learn.
- **Split**: The dataset is split into 70% training and 30% testing.
- **Cross-Validation**: 5-fold stratified cross-validation is used.
- **Models**:
  - Logistic Regression without standardization.
  - Logistic Regression with standardization.
- **Regularization**: The regularization parameter `C` is set to 1.
- **Scaling**: Standardization is performed using 